# 🕵️‍♂️ Dynamic Council Evaluation Explorer
Run the cell below to load your evaluation results and interactively read through the multi-persona reviews!

In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

results = []
try:
    with open('results_council_gemma4:26b.jsonl', 'r') as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
except FileNotFoundError:
    print("Run the dynamic_council.py script first to generate results.")
    
if results:
    correct = sum(1 for r in results if r.get('correct'))
    total = len(results)
    display(HTML(f"""
    <div style='background-color: #e8f4f8; color: #111; padding: 15px; border-radius: 8px; font-size: 18px; margin-bottom: 20px'>
        <b>Total Papers Evaluated:</b> {total} <br>
        <b>Council Accuracy:</b> {correct}/{total} ({correct/total*100:.1f}%) <br>
        <i>Based on {len(results[0].get('personas', []))} distinct persona profiles per paper</i>
    </div>
    """))

out = widgets.Output()

def view_paper(change):
    idx = change['new'] if change else 0
    if not results: return
    res = results[idx]
    gt = res['decision'].upper()
    pred = res.get('prediction', 'Unknown')
    pred = pred.upper() if pred else 'None'
    color = 'green' if res.get('correct') else '#d63031'
    
    with out:
        out.clear_output(wait=True)
        display(HTML(f"""
        <h2>📄 {res['title']}</h2>
        <h4 style='color: #666; margin-top: -10px'>OpenReview ID: <a href="{res['forum_url']}" target="_blank">{res['paper_id']}</a></h4>
        <div style='background-color: #f8f9fa; color: #111; padding: 15px; border-radius: 8px; border-left: 5px solid {color}'>
            <b>Ground Truth:</b> {gt} &nbsp;|&nbsp; 
            <b>Council Verdict:</b> <span style='color: {color}; font-weight: bold'>{pred}</span>
        </div>
        """))
        
        display(HTML("<hr><h3 style='color: #8ab4f8'>⚖️ Area Chair Consolidation Report</h3>"))
        display(Markdown(res.get('consolidation', 'No consolidation text logged.')))
        
        display(HTML("<hr><h3 style='color: #8ab4f8'>🕵️‍♂️ The Committee Reviews</h3>"))
        for rev in res.get('reviews', []):
            display(HTML("<div style='border-left: 4px solid #3498db; padding-left: 20px; padding-top: 5px; margin-bottom: 25px; background: rgba(52, 152, 219, 0.05)'>"))
            display(Markdown(rev))
            display(HTML("</div>"))

if results:
    options = [(f"[{'✓' if r.get('correct') else '✗'}] {r['title'][:60]}...", i) for i, r in enumerate(results)]
    dropdown = widgets.Dropdown(options=options, description='Select Paper:', layout={'width': '80%'}, style={'description_width': 'initial'})
    dropdown.observe(view_paper, names='value')
    view_paper(None)  # Render initial choice
    display(dropdown, out)

Dropdown(description='Select Paper:', layout=Layout(width='80%'), options=(('[✓] Retrieval-Augmented Generatio…

Output()